In [ ]:
# visualization.ipynb
# Generate plots
import pandas as pd
import numpy as np
import os
import joblib

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

sns.set_theme(style="whitegrid")

In [ ]:
# Clone repo
!git clone https://github.com/KeXen1/Student-Performance-ML-Project.git

%cd Student-Performance-ML-Project

In [ ]:
# Load cleaned dataset
data = pd.read_csv("data/processed/cleaned_data.csv")

display(data.head())
print(data.shape)

In [ ]:
# Build target and features
def grade_category(g3):
    if g3 < 10:
        return "Low"
    elif g3 < 15:
        return "Medium"
    else:
        return "High"

data["performance_category"] = data["G3"].apply(grade_category)

X = data.drop(columns=["G3", "G2", "performance_category"], errors="ignore")
y = data["performance_category"]

In [ ]:
# Output directories for plots
os.makedirs("results/plots", exist_ok=True)
os.makedirs("results/confusion_matrices", exist_ok=True)

In [ ]:
# Plot performance category distribution
category_order = ["Low", "Medium", "High"]

plt.figure(figsize=(7, 5))
sns.countplot(x=y, order=category_order, palette="viridis")
plt.title("Distribution of Student Performance Categories")
plt.xlabel("Performance Category")
plt.ylabel("Number of Students")
plt.tight_layout()
plt.savefig("results/plots/performance_distribution.png", dpi=150)
plt.show()

In [ ]:
# Correlation heatmap of numeric features
numeric_data = data.select_dtypes(include=["int64", "float64"]).copy()

plt.figure(figsize=(12, 10))
sns.heatmap(numeric_data.corr(), annot=True, fmt=".2f", cmap="coolwarm", square=True, annot_kws={"size": 7})
plt.title("Correlation Heatmap of Numeric Features")
plt.tight_layout()
plt.savefig("results/plots/correlation_heatmap.png", dpi=150)
plt.show()

In [ ]:
# Feature importance plot from saved CSV
feature_importance_df = pd.read_csv("results/feature_importance.csv")
top_features = feature_importance_df.head(15)

plt.figure(figsize=(9, 7))
sns.barplot(data=top_features, x="Importance", y="Feature", palette="viridis")
plt.title("Top 15 Feature Importances (Random Forest)")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig("results/plots/feature_importance.png", dpi=150)
plt.show()

In [ ]:
# Recreate the same train/test split used in model_training.ipynb
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=0,
    stratify=y
)

In [ ]:
# Confusion matrix plot for each model
model_files = {
    "Logistic Regression": "models/logistic_regression.pkl",
    "Decision Tree": "models/decision_tree.pkl",
    "Random Forest": "models/random_forest.pkl"
}

category_order = ["Low", "Medium", "High"]

for name, path in model_files.items():
    model = joblib.load(path)
    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred, labels=category_order)

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=category_order,
        yticklabels=category_order,
        cbar=False
    )
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted Category")
    plt.ylabel("Actual Category")
    plt.tight_layout()

    safe_name = name.lower().replace(" ", "_")
    plt.savefig(f"results/confusion_matrices/{safe_name}.png", dpi=150)
    plt.show()

In [ ]:
# Model comparison bar chart from evaluation_metrics.csv
metrics_df = pd.read_csv("results/evaluation_metrics.csv")
metrics_long = metrics_df.melt(id_vars="Model", var_name="Metric", value_name="Score")

plt.figure(figsize=(10, 6))
sns.barplot(data=metrics_long, x="Metric", y="Score", hue="Model", palette="viridis")
plt.title("Model Performance Comparison")
plt.ylim(0, 1)
plt.ylabel("Score")
plt.xlabel("Metric")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("results/plots/model_comparison.png", dpi=150)
plt.show()

In [ ]:
# Distribution of selected early-semester features by performance category
selected_numeric = ["G1", "studytime", "absences", "failures"]

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()

for ax, feature in zip(axes, selected_numeric):
    sns.boxplot(
        x=data["performance_category"],
        y=data[feature],
        order=category_order,
        palette="viridis",
        ax=ax
    )
    ax.set_title(f"{feature} by Performance Category")
    ax.set_xlabel("Performance Category")
    ax.set_ylabel(feature)

plt.tight_layout()
plt.savefig("results/plots/feature_boxplots.png", dpi=150)
plt.show()